In [2]:
import random
import pandas as pd




In [3]:


def get_five_star_rate(pity_5):
    """
    根据当前五星保底计数（垫抽数），计算单抽出五星的概率。
    第1~73抽：基础概率 0.6%
    第74抽起：每抽增加 6% 概率（软保底），直到第90抽达到100%（硬保底）。
    """
    if pity_5 < 74:  
        return 0.006
    else:
        # 第74抽概率 = 0.6% + (74-73)*6% = 6.6%
        # 第89抽概率 = 0.6% + (89-73)*6% = 96.6%
        return 0.006 + (pity_5 - 73) * 0.06



In [4]:
def single_wish(pity_5, is_guaranteed, lost_count):
    """
    执行一次单抽，返回：
        is_five: 是否出五星
        is_up:   是否出当期UP（限定五星）
        新的 pity_5
        新的 is_guaranteed
        新的 lost_count （连续小保底歪的次数，用于捕获明光）

    机制说明：
        - 软/硬保底由 get_five_star_rate 控制
        - 小保底出UP概率 50%，歪了则触发大保底（is_guaranteed=True）
        - 捕获明光：连续歪了2次后，下一次小保底必定触发（转为UP）
          同时在歪时也有小概率（此处设为10%）直接触发
    """
    # 1. 判断是否出五星
    is_five = False
    if pity_5 >= 89:          # 第90抽硬保底
        is_five = True
    else:
        rate = get_five_star_rate(pity_5)
        if random.random() < rate:
            is_five = True

    # 2. 出五星的情况
    if is_five:
        if is_guaranteed:
            # 大保底 → 必定UP
            is_up = True
            new_is_guaranteed = False
            # 出UP，重置连歪计数
            new_lost_count = 0
        else:
            # 小保底
            if random.random() < 0.5:      # 50% 概率出UP
                is_up = True
                new_is_guaranteed = False
                new_lost_count = 0
            else:
                # 歪了 → 触发大保底，并判断捕获明光
                # 如果已连续歪了2次，则本次必定触发捕获明光（强制转为UP）
                if lost_count >= 2:
                    is_up = True
                    new_lost_count = 0
                else:
                    # 否则以 10% 的概率触发捕获明光（可调整，综合UP率约55%）
                    if random.random() < 0.10:
                        is_up = True
                        new_lost_count = 0
                    else:
                        is_up = False
                        new_lost_count = lost_count + 1
                new_is_guaranteed = True   # 歪了，触发大保底
        new_pity_5 = 0                     # 重置五星保底计数

    # 3. 没出五星
    else:
        is_up = False
        new_pity_5 = pity_5 + 1
        new_is_guaranteed = is_guaranteed
        new_lost_count = lost_count        # 未出金不影响连歪计数

    return is_five, is_up, new_pity_5, new_is_guaranteed, new_lost_count

In [5]:
def simulate_one_run(start_pity, start_guaranteed, start_lost, max_pulls):
    """
    从给定状态开始，连续抽 max_pulls 次。
    返回：获得的限定五星（UP）总数。
    """
    pity = start_pity
    guaranteed = start_guaranteed
    lost = start_lost
    limited_count = 0

    for _ in range(max_pulls):
        is_five, is_up, pity, guaranteed, lost = single_wish(pity, guaranteed, lost)
        if is_five and is_up:
            limited_count += 1
        # 如果中途抽到足够多，可以提前停止？但为了统计分布，我们抽满全部

    return limited_count

In [6]:

START_PITY = 0       # 五星保底计数（0~89，即你已经垫了多少抽）
IS_GUARANTEED = False    # 是否处于大保底（True表示下一个五星必定是UP）
MAX_PULLS = 190# 继续抽的总数
SIMULATIONS = 100000     # 模拟次数（越大越精确，运行时间稍长）


print(f"初始状态：垫抽数={START_PITY}，大保底={IS_GUARANTEED}，再抽{MAX_PULLS}抽")
print("正在模拟，请稍候...")

results = []
for _ in range(SIMULATIONS):
    cnt = simulate_one_run(START_PITY, IS_GUARANTEED, 0, MAX_PULLS)  # 初始lost=0
    results.append(cnt)

# 统计每个限定金数量出现的次数
count_series = pd.Series(results).value_counts().sort_index()
prob_series = count_series / SIMULATIONS

# 构建DataFrame
df_prob = pd.DataFrame({
    '限定金数量': prob_series.index,
    '概率': prob_series.values
})

# 按概率降序排列，取前10
df_top10 = df_prob.sort_values('概率', ascending=False).head(10)

print("\n========== 模拟结果 ==========")
print(f"总模拟次数：{SIMULATIONS}")
print(f"最可能获得的限定金数量（概率最高的前10名）：")
print(df_top10.to_string(index=False))

# 额外：累计概率（至少获得n个）
print("\n===== 累计概率（至少获得n个）=====")
for n in range(0, 6):
    prob_at_least = df_prob[df_prob['限定金数量'] >= n]['概率'].sum()
    print(f"至少获得 {n} 个限定金的概率: {prob_at_least*100:.2f}%")

初始状态：垫抽数=0，大保底=False，再抽190抽
正在模拟，请稍候...

========== 模拟结果 ==========
总模拟次数：100000
最可能获得的限定金数量（概率最高的前10名）：
 限定金数量      概率
     2 0.43074
     1 0.41559
     3 0.13103
     4 0.02039
     5 0.00209
     6 0.00014
     7 0.00002

===== 累计概率（至少获得n个）=====
至少获得 0 个限定金的概率: 100.00%
至少获得 1 个限定金的概率: 100.00%
至少获得 2 个限定金的概率: 58.44%
至少获得 3 个限定金的概率: 15.37%
至少获得 4 个限定金的概率: 2.26%
至少获得 5 个限定金的概率: 0.22%
